# Unified model comparison

One metric set, computed identically for every model via `personal.shared_functions.unified_metrics`:

| metric | meaning |
|---|---|
| `ho_nse`  | **held-out** NSE = mean over strikes of \|model−mid\| / half-spread, prior (one-step-ahead) state |
| `is_nse`  | **in-sample** NSE, posterior (post-update) state |
| `ho_rmse` / `is_rmse` | same splits, RMSE in price units (C/df, dollars) |
| `neg_mass` | implied negative mass (Breeden–Litzenberger or native density); 0 = arbitrage-free |

All prices are undiscounted **call** prices C/df; puts are converted via put-call parity so every model is scored in the same space. Each model's adapter produces the *standard eval table* (`ts, strike, forward, mid_call, half_spread, prior_call, post_call`); `unified_metrics.evaluate` turns that into the metric row, and `comparison_table` stacks the rows.

In [1]:
import sys
from pathlib import Path


def _find_root(start: Path) -> Path:
    """Walk up from `start` until we find the repo root (has pyproject.toml + quantdata/)."""
    for d in (start, *start.parents):
        if (d / "pyproject.toml").exists() and (d / "quantdata").is_dir():
            return d
    raise RuntimeError(f"quant_data project root not found above {start}")


PROJECT_ROOT = _find_root(Path.cwd().resolve())

# `quantdata` is an editable-installed package, so `import quantdata...` works
# already; PROJECT_ROOT is added so it also imports without the editable install.
# The `personal/` modules are flat scripts (no __init__.py), so every directory
# we import a module from must be on sys.path explicitly.
for p in (
    PROJECT_ROOT,
    PROJECT_ROOT / "personal" / "shared_functions",
    PROJECT_ROOT / "personal" / "SANOS_KF_Strike",
    PROJECT_ROOT / "personal" / "Kalman_filter_spline",
):
    p = str(p)
    if p not in sys.path:
        sys.path.insert(0, p)

from datetime import date

import numpy as np
import pandas as pd
import polars as pl

import personal.SANOS_KF_Strike.sanos_ekf_eval as ekf

# personal/ flat modules:
import personal.shared_functions.unified_metrics as um

# quantdata package (example — import whatever you need):
import quantdata  # e.g. from quantdata.sanos import open as sanos_open
from personal.Kalman_filter_spline.surface_kalman_filters import build_iv_observations
from personal.SANOS_KF_Strike.sanos_kf_strike import (
    SanosKFStrikeConfig,
    gate_eval_table,
    run_sanos_kf_strike,
)
from quantdata.massive.client import MassiveClient

rows = []  # one metric dict per model, fed to um.comparison_table at the end

/Users/elie/Documents/GitHub/quant_data/quantdata/__init__.py:212: UserWarning: Using cache mode 'on' as specified in environment variable QUANTDATA_CACHE_MODE.


## CONFIG

In [2]:
TRADE_DATE = date(2024, 1, 3)

In [3]:
client = MassiveClient()

## 1. SANOS EKF (direct-observation, Option 2)

Runs from `phase1_cache_<date>.pkl`; snapshots are full smiles, so negative mass is computed per snapshot by Breeden–Litzenberger inside `evaluate`.

In [4]:
# ekf.load_cache("2024-03-05")  # uncomment / change to switch dates
df_ekf = ekf.eval_table(q_scale=1.0, ell=3.0, p0_scale=1e-2, warm=False, correlated=True)
m_ekf = um.evaluate(df_ekf, model_name="SANOS EKF")
rows.append(m_ekf)
m_ekf

{'model': 'SANOS EKF',
 'ho_nse_mean': 0.41229736025309577,
 'ho_nse_med': 0.36931717739769365,
 'ho_nse_std': 0.3694708301749086,
 'is_nse_mean': 0.1730049520905796,
 'is_nse_med': 0.16505717173067752,
 'is_nse_std': 0.13291329017341574,
 'ho_rmse_mean': 0.007348493103017636,
 'ho_rmse_med': 0.005757773333456527,
 'ho_rmse_std': 0.009006264586721711,
 'is_rmse_mean': 0.0046474176874496415,
 'is_rmse_med': 0.0034096748138901806,
 'is_rmse_std': 0.007868207320783627,
 'neg_mass_mean': 0.0,
 'neg_mass_med': 0.0,
 'arb_frac': 0.0,
 'n_snapshots': 24066}

## 2. SANOS 2-stage (batch DLV fit → KF on DLV²)

The two-stage incumbent, ported from `sanos_dlv_kf_eval_lean.ipynb` with **its exact hyperparameters**:

- **Stage 1 — batch DLV fit.** At each `BATCH_FREQ` bucket, run the SANOS absolute-strike optimizer (`run_intraday_loop` / `fit_density`) to get a DLV² vector per snapshot.
- **Stage 2 — correlated-Q Kalman filter on DLV²** (`run_dlv_kalman_filter_v2`, `KFConfig`). Random-walk transition, cross-strike RBF process noise, selection-H for stale nodes.

`prior_call` is priced from the KF **prior** state `x_pred_hist[k]` (one-step-ahead, held-out); `post_call` from the KF **posterior** `x_hist[k]` (in-sample). Each step's DLV² is turned back into a density via the N×N exactly-determined linear system (`_density_from_kf_dlvsq`), priced with `option_prices`, then fed through `um.from_option_prices` / `um.evaluate`. Negative mass is computed per snapshot by Breeden–Litzenberger on the posterior curve.

Runs on the unified notebook's `TRADE_DATE` (same data as the SANOS EKF above) so the comparison is apples-to-apples; only the **hyperparameters** match the gate's eval notebook.

In [5]:
df_tickers = pl.read_csv(f"spy_options_tickers_{TRADE_DATE}.csv")

df_spy = client.fetch_trades("SPY", TRADE_DATE)

In [6]:
df_quotes, df_trades = build_iv_observations(df_tickers, TRADE_DATE, df_spy, min_trades=100)

Skipping O:SPY240105C00481000: 46 bad quotes
Skipping O:SPY240105C00482000: 64 bad quotes
Skipping O:SPY240105C00483000: 842 bad quotes
Skipping O:SPY240105C00484000: 2129 bad quotes
Skipping O:SPY240105C00485000: 3888 bad quotes
Skipping O:SPY240105P00462500: 17 bad quotes
Expiry 2024-01-05: implied forward from 13 strike(s)


In [7]:
# ===== Stage 1: batch DLV fit (same pipeline & config as sanos_dlv_kf_eval_lean.ipynb) =====
from functools import partial

from dlv_fit_absolute import fit_density
from dlv_grid_absolute import make_density0, make_strike_grid
from dlv_pricing_absolute import option_prices
from dlv_snapshot_absolute import make_batch_snapshots, run_intraday_loop

# --- DLV grid / fit config (lean-notebook values) ---
BATCH_FREQ = "1s"  # "1s" | "1m" | "5m"  (1s = faithful to the gate eval; coarser = much faster)
LOWER_SCALE, UPPER_SCALE, N_PAD = 0.9, 1.1, 1
DLV_MIN, DLV_MAX = 0.001, 2.0
CALLS_ONLY, PUTS_ONLY = False, True
# Single subset switch reused by every model's fit+eval below so the comparison is
# apples-to-apples. None => both calls and puts; here it resolves to puts only.
EVAL_IS_CALL = CALLS_ONLY if (CALLS_ONLY or PUTS_ONLY) else None
REGULARIZER_WEIGHT = 0.0
LOG_DLV_SPACE = False
TEMPORAL_REGULARIZER_WEIGHT = 0.0
TEMPORAL_WEIGHT_CLIP = float("inf")
BID_ASK_PENALTY_WEIGHT = 0.0

# Normalise prices by discount factor (absolute-strike convention: C/df, P/df)
_has_df = "discount_factor" in df_quotes.columns
_df_scale = pl.col("discount_factor") if _has_df else pl.lit(1.0)
df_q = df_quotes.drop_nulls().with_columns(
    (pl.col("bid_price") / _df_scale).alias("bid_norm"),
    (pl.col("ask_price") / _df_scale).alias("ask_norm"),
    ((pl.col("bid_price") + pl.col("ask_price")) / 2.0 / _df_scale).alias("mid_norm"),
    pl.col("strike").alias("pure_strike"),
)
df_calls = df_q.filter(pl.col("is_call"))
df_puts = df_q.filter(~pl.col("is_call"))
tte_y_max = float(df_q["T"].max())
tte_y_med = float(df_q["T"].median())

# Strike grid + initial density
tte_geom = tte_y_max * 1.1
df_anchor = df_puts if PUTS_ONLY else df_calls
ref_fwd = float(df_anchor["forward"].median())
obs_ks = np.sort(df_anchor["strike"].unique().to_numpy().astype(float))
grid = make_strike_grid(
    obs_ks, fwd=ref_fwd, lower_scale=LOWER_SCALE, upper_scale=UPPER_SCALE, n_pad=N_PAD
)
xstrikes_row = grid["xstrikes_row"]
anchor_pure = grid["anchor_pure"]
grid_K, density0 = make_density0(xstrikes_row, fwd=ref_fwd)
dlv_anchor_pure = anchor_pure[1:-1]  # (N-2,) strikes carrying the DLV² state
N_dlv = len(dlv_anchor_pure)

_fit_fn = partial(
    fit_density,
    xstrikes_row=xstrikes_row,
    grid_K=grid_K,
    density0=density0,
    tte_geom=tte_geom,
    dlv_min=DLV_MIN,
    dlv_max=DLV_MAX,
    regularizer_weight=REGULARIZER_WEIGHT,
    log_dlv_space=LOG_DLV_SPACE,
    temporal_regularizer_weight=TEMPORAL_REGULARIZER_WEIGHT,
    temporal_weight_clip=TEMPORAL_WEIGHT_CLIP,
    bid_ask_penalty_weight=BID_ASK_PENALTY_WEIGHT,
)

# Batch snapshots + intraday DLV fit
snaps_calls = make_batch_snapshots(df_calls, BATCH_FREQ)
snaps_puts = make_batch_snapshots(df_puts, BATCH_FREQ)
snap_ref = snaps_puts if PUTS_ONLY else snaps_calls
batch_ts_list = sorted(snap_ref["batch_ts"].unique().to_list())

try:
    from tqdm.auto import tqdm
except ImportError:

    def tqdm(x, **kw):
        return x


results = run_intraday_loop(
    batch_ts_list=tqdm(batch_ts_list, desc="Fitting DLV snapshots"),
    snaps_calls=snaps_calls,
    snaps_puts=snaps_puts,
    fit_fn=_fit_fn,
    tte_y_med=tte_y_med,
    calls_only=CALLS_ONLY,
    puts_only=PUTS_ONLY,
    verbose=False,
    anchor_pure=anchor_pure,
)
n_failed = sum(r["failed"] for r in results)
print(f"Stage 1 done — {len(results):,} snapshots ({N_dlv} DLV² dims), {n_failed} failed")

Stage 1 done — 24,066 snapshots (21 DLV² dims), 0 failed


In [8]:
# ===== Stage 2: correlated-Q Kalman filter on DLV² (lean-notebook KF_* hyperparameters) =====
from dlv_kalman_filter import KFConfig, run_dlv_kalman_filter_v2

# Kalman filter hyperparameters — identical to sanos_dlv_kf_eval_lean.ipynb
KF_Q_SCALE = 1e5  # process noise variance per trading day (per DLV² component)
KF_R_DIAG = 1e-15  # observation noise variance for fresh nodes (r_fresh)
KF_R_STALE = 1e-15  # observation noise for stale nodes; inf = selection-H
KF_P0_SCALE = 1e-6  # initial posterior covariance diagonal
TRADING_DAY_S = 23400.0  # 6.5 h in seconds
KF_IGNORE_AT_DLV_MAX = True
KF_ELL = 3.0  # RBF correlation length (dollars)
KF_STENCIL = 3  # 1 = permissive, 3 = conservative fresh-node rule
KF_P0_CORRELATED = True
KF_SOFT_FLOOR = 0.0

cfg_v2 = KFConfig(
    q_scale=KF_Q_SCALE,
    ell=KF_ELL,
    r_diag=KF_R_DIAG,
    r_stale=KF_R_STALE,
    p0_scale=KF_P0_SCALE,
    p0_correlated=KF_P0_CORRELATED,
    stencil=KF_STENCIL,
    soft_floor=KF_SOFT_FLOOR,
    trading_day_s=TRADING_DAY_S,
    ignore_at_dlv_max=KF_IGNORE_AT_DLV_MAX,
    dlv_max_sq_net=DLV_MAX**2 - DLV_MIN**2,
)
x_hist, x_pred_hist, P_diag_hist, innov_hist, ts_hist, failed_hist, n_fresh_hist = (
    run_dlv_kalman_filter_v2(results, N_dlv=N_dlv, grid_K_nodes=dlv_anchor_pure, cfg=cfg_v2)
)
print(f"Stage 2 done — KF over {len(results):,} steps, {int(failed_hist.sum())} failed")


# ── KF DLV² state → density (N×N exactly-determined system) → standard eval table ──
_kf_N = len(grid_K)
_kf_box = grid_K**2 / (xstrikes_row[2:] - xstrikes_row[:-2]) * tte_geom
_kf_box[0] = 0.0
_kf_box[-1] = 0.0
_kf_pay = np.maximum(grid_K[np.newaxis, :] - grid_K[:, np.newaxis], 0.0)  # pay[j,i]=max(K_i-K_j,0)


def _density_from_kf_dlvsq(dlvsq_net, fwd, dlv_min=DLV_MIN):
    """Reconstruct density from (N-2,) net-DLV²; None if solve fails or >2% neg. mass."""
    N = _kf_N
    v = dlvsq_net + dlv_min**2
    A = np.zeros((N, N))
    b = np.zeros(N)
    A[0] = 1.0
    b[0] = 1.0  # sum d = 1
    A[1] = grid_K
    b[1] = fwd  # K @ d = fwd
    for j in range(1, N - 1):
        A[j + 1] = _kf_pay[j]
        A[j + 1, j] -= v[j - 1] * _kf_box[j]
        b[j + 1] = max(fwd - grid_K[j], 0.0)
    try:
        d = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        return None
    if (-np.minimum(d, 0.0)).sum() > 0.02:
        return None
    d = np.maximum(d, 0.0)
    s = d.sum()
    return d / s if s > 1e-12 else None


# prior = x_pred_hist[k] (held-out, pre-update); post = x_hist[k] (in-sample, post-update)
_snap_src = snaps_puts if PUTS_ONLY else snaps_calls
_is_call_flag = not PUTS_ONLY
_rec = {k: [] for k in ("ts", "strike", "forward", "is_call", "bid", "ask", "prior", "post")}

for k, r in enumerate(results):
    if r["failed"]:
        continue
    snap = _snap_src.filter(pl.col("batch_ts") == r["ts"]).sort("pure_strike")
    if snap.is_empty():
        continue
    strikes_k = snap["pure_strike"].to_numpy().astype(float)
    bid_k = snap["bid_norm"].to_numpy().astype(float)
    ask_k = snap["ask_norm"].to_numpy().astype(float)
    fwd_k = r["fwd"]
    is_call_k = np.full(len(strikes_k), _is_call_flag, dtype=bool)

    d_post = _density_from_kf_dlvsq(x_hist[k], fwd_k)
    d_prior = _density_from_kf_dlvsq(x_pred_hist[k], fwd_k)
    if d_post is None or d_prior is None:
        continue
    px_post = option_prices(d_post, grid_K, strikes_k, is_call_k, fwd_k)
    px_prior = option_prices(d_prior, grid_K, strikes_k, is_call_k, fwd_k)

    n = len(strikes_k)
    _rec["ts"].extend([k] * n)
    _rec["strike"].extend(strikes_k)
    _rec["forward"].extend([fwd_k] * n)
    _rec["is_call"].extend(is_call_k)
    _rec["bid"].extend(bid_k)
    _rec["ask"].extend(ask_k)
    _rec["prior"].extend(px_prior)
    _rec["post"].extend(px_post)

# bid_norm/ask_norm and model prices are already undiscounted (C/df, P/df) → discount_factor=1
df_gate = um.from_option_prices(
    ts=_rec["ts"],
    strike=_rec["strike"],
    forward=_rec["forward"],
    is_call=_rec["is_call"],
    bid=_rec["bid"],
    ask=_rec["ask"],
    prior_price=_rec["prior"],
    post_price=_rec["post"],
)
m_gate = um.evaluate(df_gate, model_name="SANOS 2-stage (DLV fit + KF)")
rows.append(m_gate)
m_gate

Stage 2 done — KF over 24,066 steps, 0 failed


{'model': 'SANOS 2-stage (DLV fit + KF)',
 'ho_nse_mean': 1.7182574790230025,
 'ho_nse_med': 0.5965504399955517,
 'ho_nse_std': 4.562455145241554,
 'is_nse_mean': 0.43616884751573126,
 'is_nse_med': 0.00020374218686011438,
 'is_nse_std': 3.065843848086008,
 'ho_rmse_mean': 0.02136950382257551,
 'ho_rmse_med': 0.008930966561078772,
 'ho_rmse_std': 0.04313794031031083,
 'is_rmse_mean': 0.005096660343457375,
 'is_rmse_med': 1.8367476463728317e-06,
 'is_rmse_std': 0.02613419718424606,
 'neg_mass_mean': 0.0,
 'neg_mass_med': 0.0,
 'arb_frac': 0.0,
 'n_snapshots': 24066}

## 3. KF on splines / parameters

Same methodology as the SANOS models: **fit in 1-second batches** (`run_spline_surface_kalman(..., batch_seconds=1.0)` — one time-proportional predict + one batch update per window) and **evaluate per 1-second cross-sectional snapshot**.

Within a window the shared state θ gives every quote one held-out (previous-window) prior and one posterior, so we keep the **last quote of each (strike, is_call) per window** as the snapshot representative and group by the window index — exactly like the SANOS 2-stage table (`ts = batch`, one row per (snapshot, strike)). `n_snapshots` is therefore the number of 1 s buckets (~24k), not the raw tick count.

`prior_price` = `kalman_price_prior` (pre-update / held-out), `post_price` = `kalman_price` (post-update). Negative mass: reconstruct the smile across a strike grid at sampled times, price with `bs_call_price`, then `negative_mass_surface`.

In [9]:
import personal.Kalman_filter_spline.surface_model as sm

# df_quotes_iv / df_trades_iv are the IV-annotated outputs of build_iv_observations
# (already computed above as df_quotes / df_trades — just alias them here)
df_quotes_iv = df_quotes
df_trades_iv = df_trades
if EVAL_IS_CALL is not None:  # fit + eval on the same subset as the SANOS models (puts only)
    df_quotes_iv = df_quotes_iv.filter(pl.col("is_call") == EVAL_IS_CALL)
    df_trades_iv = df_trades_iv.filter(pl.col("is_call") == EVAL_IS_CALL)

# Spline surface KF config
# SVI_SLICE_MODEL: 5-param Gatheral SVI, single-expiry (good for short-dated SPY)
# Swap for sm.QUAD_SLICE_MODEL (3 params) or sm.BSPLINE_SLICE_MODEL (8 params) to compare
cfg = sm.SurfaceModelConfig(
    model=sm.SVI_SLICE_MODEL,
    Q_scale=1e-4,  # process noise PER SECOND (batched fit ⇒ P_pred = P + Q·Δt); retune vs per-tick
    Q_structure="diagonal",  # per-param scaling via model.q_diag_weights
    P0_scale=1e-3,  # initial uncertainty (large → flat start, KF converges quickly)
)

In [10]:
from personal.Kalman_filter_spline.surface_kalman_filters import run_spline_surface_kalman

BATCH_S = 1.0  # fit & evaluate in 1-second batches (same methodology as the SANOS models)

q_res, t_res, df_state = run_spline_surface_kalman(
    df_quotes_iv, df_trades_iv, cfg, batch_seconds=BATCH_S
)
q = q_res.sort(["strike", "is_call", "ts_ns"]).to_pandas()

# 1 s window id; within a window the spline shares one posterior θ and one held-out (previous-
# window) prior θ, so prices vary only through each quote's own forward/T/k. Take the LAST quote
# of each (strike, is_call) in the window as the per-second snapshot representative, then group by
# the window — one cross-sectional snapshot per 1 s bucket, exactly like the SANOS 2-stage table.
q["window"] = q["ts_ns"].astype("int64") // int(BATCH_S * 1e9)
# snap = q.sort_values("ts_ns").drop_duplicates(subset=["window", "strike", "is_call"], keep="last")
# held-out prior is the batched filter's own pre-update prediction; drop any row where it is
# undefined (e.g. invalid forward/T at the snapshot quote).
snap = q
snap = snap.dropna(subset=["kalman_price_prior"])

df_spline = um.from_option_prices(
    ts=snap["window"],  # → n_snapshots = number of 1 s buckets (~24k), cross-sectional over strikes
    strike=snap["strike"],
    forward=snap["forward"],
    is_call=snap["is_call"],
    bid=snap["bid_price"],
    ask=snap["ask_price"],
    prior_price=snap["kalman_price_prior"],
    post_price=snap["kalman_price"],
    discount_factor=snap["discount_factor"] if "discount_factor" in snap else None,
)

In [11]:
q_res

ts_ns,strike,is_call,log_moneyness,T,observed_iv,bid_iv,ask_iv,R_iv,bid_price,ask_price,forward,discount_factor,spline_iv,spline_prior_iv,kalman_price,kalman_price_prior
"datetime[ns, America/New_York]",f64,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2024-01-03 09:30:00.008294144 EST,471.0,false,0.001211,0.006217,0.127732,0.126371,0.129092,0.000002,2.17,2.21,470.43,null,0.042173,0.155109,0.950408,2.592922
2024-01-03 09:30:00.008504576 EST,466.0,false,-0.009462,0.006217,0.141883,0.140865,0.142848,9.8379e-7,0.57,0.59,470.43,null,0.052484,0.15709,0.007384,0.742633
2024-01-03 09:30:00.008594688 EST,465.0,false,-0.01161,0.006217,0.14471,0.143565,0.145845,0.000001,0.42,0.44,470.43,null,0.054604,0.157503,0.002159,0.547026
2024-01-03 09:30:00.008800256 EST,462.0,false,-0.018082,0.006217,0.152948,0.145223,0.159941,0.000054,0.13,0.2,470.43,null,0.061084,0.158777,0.000046,0.193956
2024-01-03 09:30:00.008815872 EST,463.0,false,-0.01592,0.006217,0.149568,0.143385,0.15533,0.000036,0.19,0.26,470.43,null,0.058904,0.158347,0.000169,0.279654
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2024-01-03 16:14:56.120656640 EST,474.0,false,0.010487,0.005447,0.140719,0.132673,0.143631,0.00003,5.28,5.37,469.055,1.0,0.001355,2.20063,4.945,33.055485
2024-01-03 16:14:59.884593152 EST,474.0,false,0.010487,0.005447,0.138451,0.132675,0.144803,0.000037,5.28,5.38,469.055,1.0,0.243915,0.001355,6.417767,4.945
2024-01-03 16:14:59.884598272 EST,472.0,false,0.006259,0.005447,0.13324,0.131454,0.136743,0.000007,3.66,3.72,469.055,1.0,0.243915,0.001355,5.053899,2.945


In [12]:
q

,ts_ns,strike,is_call,log_moneyness,T,observed_iv,bid_iv,ask_iv,R_iv,bid_price,ask_price,forward,discount_factor,spline_iv,spline_prior_iv,kalman_price,kalman_price_prior,window
0,2024-01-03 09:30:00.165348352-05:00,450.0,False,-0.045048,0.006217,0.245402,0.233628,0.254767,0.000112,0.02,0.04,470.7350,1.0,0.089524,0.164516,4.262106e-11,3.890696e-04,1704292200
1,2024-01-03 09:30:00.249509376-05:00,450.0,False,-0.045069,0.006217,0.246281,0.245502,0.254871,0.000022,0.03,0.04,470.7450,1.0,0.089547,0.164521,4.225057e-11,3.867280e-04,1704292200
2,2024-01-03 09:30:00.456362240-05:00,450.0,False,-0.045074,0.006217,0.245810,0.245528,0.254897,0.000022,0.03,0.04,470.7475,1.0,0.089553,0.164522,4.215848e-11,3.861447e-04,1704292200
3,2024-01-03 09:30:00.522454528-05:00,450.0,False,-0.045133,0.006217,0.250240,0.245805,0.255182,0.000022,0.03,0.04,470.7750,1.0,0.089617,0.164535,4.115922e-11,3.797837e-04,1704292200
4,2024-01-03 09:30:01.063666688-05:00,450.0,False,-0.045149,0.006217,0.248158,0.245881,0.255260,0.000022,0.03,0.04,470.7825,1.0,0.015499,0.089635,6.752628e-301,4.089103e-11,1704292201
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1159106,2024-01-03 16:14:38.426663168-05:00,477.0,False,0.016812,0.005448,0.186185,0.152323,0.212438,0.000903,8.11,8.49,469.0475,1.0,0.712016,0.018930,1.439547e+01,7.952500e+00,1704316478
1159107,2024-01-03 16:14:38.872359680-05:00,477.0,False,0.016812,0.005448,0.181682,0.152324,0.211159,0.000865,8.11,8.48,469.0475,1.0,0.712016,0.018930,1.439547e+01,7.952500e+00,1704316478
1159108,2024-01-03 16:14:54.423237376-05:00,477.0,False,0.016850,0.005447,0.157488,0.148631,0.209223,0.000918,8.11,8.48,469.0300,1.0,0.194620,0.001355,8.372725e+00,7.970000e+00,1704316494
1159109,2024-01-03 16:14:54.460962560-05:00,477.0,False,0.016844,0.005447,0.174954,0.136282,0.209503,0.001340,8.06,8.48,469.0325,1.0,0.194620,0.001355,8.370530e+00,7.967500e+00,1704316494


In [13]:
df_spline

,ts,strike,forward,mid_call,half_spread,prior_call,post_call
0,1704292200,450.0,470.7350,20.7650,0.010,20.735389,20.735000
1,1704292200,450.0,470.7450,20.7800,0.005,20.745387,20.745000
2,1704292200,450.0,470.7475,20.7825,0.005,20.747886,20.747500
3,1704292200,450.0,470.7750,20.8100,0.005,20.775380,20.775000
4,1704292201,450.0,470.7825,20.8175,0.005,20.782500,20.782500
...,...,...,...,...,...,...,...
1159106,1704316478,477.0,469.0475,0.3475,0.190,0.000000,6.442971
1159107,1704316478,477.0,469.0475,0.3425,0.185,0.000000,6.442971
1159108,1704316494,477.0,469.0300,0.3250,0.185,0.000000,0.402725
1159109,1704316494,477.0,469.0325,0.3025,0.210,0.000000,0.403030


In [14]:
m_spline = um.evaluate(df_spline, model_name="KF spline", neg_mass_values=None)
rows.append(m_spline)
m_spline

{'model': 'KF spline',
 'ho_nse_mean': 455.0649374567995,
 'ho_nse_med': 2.1007959519258406,
 'ho_nse_std': 1681.5968546667523,
 'is_nse_mean': 460.57011387053853,
 'is_nse_med': 1.8937578314058938,
 'is_nse_std': 1687.718089903146,
 'ho_rmse_mean': 3.741366182669194,
 'ho_rmse_med': 0.04664291520039218,
 'ho_rmse_std': 13.19935683363997,
 'is_rmse_mean': 3.75433372403437,
 'is_rmse_med': 0.044583978587334754,
 'is_rmse_std': 13.21169723213794,
 'neg_mass_mean': -0.8150883420921287,
 'neg_mass_med': 0.0,
 'arb_frac': 0.14072738466951054,
 'n_snapshots': 24066}

In [15]:
# ── Negative mass: reconstruct smile over a strike grid at sampled snapshot times ──
state_pd = df_state.to_pandas().sort_values("ts_ns").reset_index(drop=True)
n_params = cfg.n_params
theta_cols = [f"theta_{j}" for j in range(n_params)]

# Strike grid spanning the observed strikes
K_lo = float(df_quotes_iv["strike"].min())
K_hi = float(df_quotes_iv["strike"].max())
grid_K = np.linspace(K_lo, K_hi, 41)

# Subsample snapshot times for speed (1 000 is usually plenty)
n_snap = min(1000, len(state_pd))
samp_idx = np.linspace(0, len(state_pd) - 1, n_snap, dtype=int)
state_samp = state_pd.iloc[samp_idx].reset_index(drop=True)

# For each sampled ts: look up the most-recent forward/T from the quote stream
q_sorted = q.sort_values("ts_ns").reset_index(drop=True)
asof = np.searchsorted(q_sorted["ts_ns"].values, state_samp["ts_ns"].values, side="right") - 1
asof = np.clip(asof, 0, len(q_sorted) - 1)
F_vec = q_sorted["forward"].values[asof]
T_vec = q_sorted["T"].values[asof]
thetas_samp = state_samp[theta_cols].values  # (n_snap, n_params)

# Reconstruct call-price matrix (n_snap, n_grid)
C_mat = np.full((n_snap, len(grid_K)), np.nan)
for i in range(n_snap):
    theta_i, F_i, T_i = thetas_samp[i], F_vec[i], T_vec[i]
    if not (np.isfinite(theta_i).all() and F_i > 0 and np.isfinite(F_i) and T_i > 0):
        continue
    ivs = np.array([cfg.model.observe(theta_i, np.log(k / F_i), T_i)[0] for k in grid_K])
    C_mat[i] = um.bs_call_price(np.full(len(grid_K), F_i), grid_K, np.full(len(grid_K), T_i), ivs)

valid = np.isfinite(C_mat).all(axis=1)
nm = um.negative_mass_surface(grid_K, C_mat[valid], F_vec[valid])

m_spline = um.evaluate(df_spline, model_name="KF spline", neg_mass_values=nm)
rows.append(m_spline)
m_spline

{'model': 'KF spline',
 'ho_nse_mean': 455.0649374567995,
 'ho_nse_med': 2.1007959519258406,
 'ho_nse_std': 1681.5968546667523,
 'is_nse_mean': 460.57011387053853,
 'is_nse_med': 1.8937578314058938,
 'is_nse_std': 1687.718089903146,
 'ho_rmse_mean': 3.741366182669194,
 'ho_rmse_med': 0.04664291520039218,
 'ho_rmse_std': 13.19935683363997,
 'is_rmse_mean': 3.75433372403437,
 'is_rmse_med': 0.044583978587334754,
 'is_rmse_std': 13.21169723213794,
 'neg_mass_mean': -0.02997302691033875,
 'neg_mass_med': 0.0,
 'arb_frac': 0.129,
 'n_snapshots': 24066}

## 4. KF on single strikes

Random-walk EKF (`f(x)=x`), so the one-step-ahead **prior == previous posterior**: `prior_price` is `kalman_price` shifted one quote back *within each strike*. `build_iv_surface` returns a per-strike `kalman_price`.

In [16]:
from personal.shared_functions.kalman_filters import build_iv_surface

# is_call_filter=EVAL_IS_CALL (set in the DLV cell): same subset as SANOS (puts only)

df_quotes_iv, df_trades_iv = build_iv_surface(
    df_tickers,
    TRADE_DATE,
    df_spy=df_spy,
    min_trades=100,
    simple=False,
    is_call_filter=EVAL_IS_CALL,
)


q = df_quotes_iv.sort(["strike", "is_call", "ts_ns"]).to_pandas()
# one-step-ahead prior = previous posterior within each (strike, is_call)
q["kalman_price_prior"] = q.groupby(["strike", "is_call"])["kalman_price"].shift(1)
q = q.dropna(subset=["kalman_price_prior"])
df_single = um.from_option_prices(
    ts=q["ts_ns"],
    strike=q["strike"],
    forward=q["forward"],
    is_call=q["is_call"],
    bid=q["bid_price"],
    ask=q["ask_price"],
    prior_price=q["kalman_price_prior"],
    post_price=q["kalman_price"],
    discount_factor=q["discount_factor"] if "discount_factor" in q else None,
)

O:SPY240105C00469000: dropping 2 bad quotes (crossed×2 (first=2024-01-03 11:49:45.633413-05:00))
O:SPY240105C00472000: dropping 1 bad quotes (crossed×1 (first=2024-01-03 15:51:02.275318-05:00))
O:SPY240105C00473000: dropping 1 bad quotes (crossed×1 (first=2024-01-03 10:29:17.744517-05:00))
O:SPY240105C00474000: dropping 4 bad quotes (crossed×4 (first=2024-01-03 10:25:56.925706-05:00))
O:SPY240105C00475000: dropping 1 bad quotes (crossed×1 (first=2024-01-03 14:02:06.496803-05:00))
O:SPY240105C00477000: dropping 1 bad quotes (crossed×1 (first=2024-01-03 10:00:44.569057-05:00))
O:SPY240105C00481000: dropping 46 bad quotes (sub-tick bid×46 (first=2024-01-03 16:00:02.443177-05:00))
O:SPY240105C00482000: dropping 64 bad quotes (sub-tick bid×64 (first=2024-01-03 10:00:01.421753-05:00))
O:SPY240105C00483000: dropping 842 bad quotes (sub-tick bid×842 (first=2024-01-03 09:30:00.060278-05:00), zero bid size×1 (first=2024-01-03 09:30:00.060278-05:00))
O:SPY240105C00484000: dropping 2129 bad quotes

In [17]:
# ── Negative mass for single-strike KF ──
# Use post_call from df_single directly (already undiscounted call prices via put-call parity)
# Keep only puts to avoid two quote-streams per strike
puts_mask = ~q["is_call"]  # q still has is_call; filter ts values that came from puts
ds = df_single[df_single["ts"].isin(q.loc[puts_mask, "ts_ns"])].copy()

unique_K_single = np.sort(ds["strike"].unique())

n_snap_single = min(500, ds["ts"].nunique())
ts_int = ds["ts"].astype(np.int64).values
snap_ts_int = np.linspace(ts_int.min(), ts_int.max(), n_snap_single, dtype=np.int64)

strike_groups_s = {
    K: grp.sort_values("ts").reset_index(drop=True) for K, grp in ds.groupby("strike")
}

# Forward from reference strike
ref_K_s = unique_K_single[len(unique_K_single) // 2]
ref_grp_s = strike_groups_s[ref_K_s]
ref_ts_int = ref_grp_s["ts"].astype(np.int64).values
ref_asof_s = np.clip(
    np.searchsorted(ref_ts_int, snap_ts_int, side="right") - 1, 0, len(ref_grp_s) - 1
)
F_arr_s = ref_grp_s["forward"].values[ref_asof_s]

C_mat_s = np.full((n_snap_single, len(unique_K_single)), np.nan)
for j, K_j in enumerate(unique_K_single):
    grp = strike_groups_s[K_j]
    grp_ts_int = grp["ts"].astype(np.int64).values
    idx = np.clip(np.searchsorted(grp_ts_int, snap_ts_int, side="right") - 1, 0, len(grp) - 1)
    C_mat_s[:, j] = grp["post_call"].values[idx]

valid_s = np.isfinite(C_mat_s).all(axis=1) & np.isfinite(F_arr_s)
nm_single = um.negative_mass_surface(unique_K_single, C_mat_s[valid_s], F_arr_s[valid_s])

In [18]:
# negative mass: at sampled times, gather kalman_iv across all strikes (asof / pivot),
# price with um.bs_call_price -> C_mat, then um.negative_mass_surface(grid_K, C_mat, F_vec)
m_single = um.evaluate(df_single, model_name="KF single-strike", neg_mass_values=nm_single)
rows.append(m_single)
m_single

{'model': 'KF single-strike',
 'ho_nse_mean': 0.3155097093584057,
 'ho_nse_med': 0.19985497563193352,
 'ho_nse_std': 0.42568920593647586,
 'is_nse_mean': 0.24120782889506714,
 'is_nse_med': 0.18316849932505935,
 'is_nse_std': 0.2329759665772495,
 'ho_rmse_mean': 0.00602119418293762,
 'ho_rmse_med': 0.0020049359246741005,
 'ho_rmse_std': 0.018334186784726505,
 'is_rmse_mean': 0.004881935301023051,
 'is_rmse_med': 0.0014878051113669244,
 'is_rmse_std': 0.015071814352825497,
 'neg_mass_mean': -0.2587846358643581,
 'neg_mass_med': -0.20117890981298878,
 'arb_frac': 0.994,
 'n_snapshots': 1169635}

## 5. KF on single strikes — 1-second batches

Same per-strike random-walk EKF as §4, but quotes are grouped into 1-second windows and
assimilated with **one batch update per window** (`build_iv_surface(..., batch_seconds=1.0)`).
Process noise is time-proportional (`q_scale` per second), so a burst of quotes no longer
inflates the state by quote count.

Within a 1 s window every quote of a given (strike, is_call) shares the same posterior IV, so
the held-out one-step-ahead prior is the **previous window's** posterior — not the previous
quote's (that would leak in-sample info for every quote after the first in a window). `T` is
not returned by `build_iv_surface`, so — exactly like §4 — the prior is the previous window's
posterior *price* (≤1 s of forward/T drift, negligible for same-day SPY).

In [19]:
BATCH_S = 1.0  # window width (seconds) - must match batch_seconds below

df_quotes_iv_b, df_trades_iv_b = build_iv_surface(
    df_tickers,
    TRADE_DATE,
    df_spy=df_spy,
    min_trades=100,
    simple=False,
    batch_seconds=BATCH_S,
    q_scale=0.001,
    is_call_filter=EVAL_IS_CALL,  # same subset as the SANOS models (puts only)
)

qb = df_quotes_iv_b.sort(["strike", "is_call", "ts_ns"]).to_pandas()
# 1 s window id; posterior IV is constant within (strike, is_call, window)
qb["window"] = qb["ts_ns"].astype("int64") // int(BATCH_S * 1e9)
# held-out prior = previous window's posterior price (representative = last quote in window)
_post_w = qb.groupby(["strike", "is_call", "window"], as_index=False).agg(
    _post_win=("kalman_price", "last")
)
_post_w["kalman_price_prior"] = _post_w.groupby(["strike", "is_call"])["_post_win"].shift(1)
qb = qb.merge(
    _post_w[["strike", "is_call", "window", "kalman_price_prior"]],
    on=["strike", "is_call", "window"],
    how="left",
)
qb = qb.dropna(subset=["kalman_price_prior"])
df_single_batch = um.from_option_prices(
    ts=qb["ts_ns"],
    strike=qb["strike"],
    forward=qb["forward"],
    is_call=qb["is_call"],
    bid=qb["bid_price"],
    ask=qb["ask_price"],
    prior_price=qb["kalman_price_prior"],
    post_price=qb["kalman_price"],
    discount_factor=qb["discount_factor"] if "discount_factor" in qb else None,
)

O:SPY240105C00469000: dropping 2 bad quotes (crossed×2 (first=2024-01-03 11:49:45.633413-05:00))
O:SPY240105C00472000: dropping 1 bad quotes (crossed×1 (first=2024-01-03 15:51:02.275318-05:00))
O:SPY240105C00473000: dropping 1 bad quotes (crossed×1 (first=2024-01-03 10:29:17.744517-05:00))
O:SPY240105C00474000: dropping 4 bad quotes (crossed×4 (first=2024-01-03 10:25:56.925706-05:00))
O:SPY240105C00475000: dropping 1 bad quotes (crossed×1 (first=2024-01-03 14:02:06.496803-05:00))
O:SPY240105C00477000: dropping 1 bad quotes (crossed×1 (first=2024-01-03 10:00:44.569057-05:00))
O:SPY240105C00481000: dropping 46 bad quotes (sub-tick bid×46 (first=2024-01-03 16:00:02.443177-05:00))
O:SPY240105C00482000: dropping 64 bad quotes (sub-tick bid×64 (first=2024-01-03 10:00:01.421753-05:00))
O:SPY240105C00483000: dropping 842 bad quotes (sub-tick bid×842 (first=2024-01-03 09:30:00.060278-05:00), zero bid size×1 (first=2024-01-03 09:30:00.060278-05:00))
O:SPY240105C00484000: dropping 2129 bad quotes

In [20]:
# ── Negative mass for batched single-strike KF (mirror of §4) ──
puts_mask_b = ~qb["is_call"]
ds_b = df_single_batch[df_single_batch["ts"].isin(qb.loc[puts_mask_b, "ts_ns"])].copy()

unique_K_b = np.sort(ds_b["strike"].unique())

n_snap_b = min(500, ds_b["ts"].nunique())
ts_int_b = ds_b["ts"].astype(np.int64).values
snap_ts_int_b = np.linspace(ts_int_b.min(), ts_int_b.max(), n_snap_b, dtype=np.int64)

strike_groups_b = {
    K: grp.sort_values("ts").reset_index(drop=True) for K, grp in ds_b.groupby("strike")
}

ref_K_b = unique_K_b[len(unique_K_b) // 2]
ref_grp_b = strike_groups_b[ref_K_b]
ref_ts_int_b = ref_grp_b["ts"].astype(np.int64).values
ref_asof_b = np.clip(
    np.searchsorted(ref_ts_int_b, snap_ts_int_b, side="right") - 1, 0, len(ref_grp_b) - 1
)
F_arr_b = ref_grp_b["forward"].values[ref_asof_b]

C_mat_b = np.full((n_snap_b, len(unique_K_b)), np.nan)
for j, K_j in enumerate(unique_K_b):
    grp = strike_groups_b[K_j]
    grp_ts_int = grp["ts"].astype(np.int64).values
    idx = np.clip(np.searchsorted(grp_ts_int, snap_ts_int_b, side="right") - 1, 0, len(grp) - 1)
    C_mat_b[:, j] = grp["post_call"].values[idx]

valid_b = np.isfinite(C_mat_b).all(axis=1) & np.isfinite(F_arr_b)
nm_batch = um.negative_mass_surface(unique_K_b, C_mat_b[valid_b], F_arr_b[valid_b])

In [21]:
m_single_batch = um.evaluate(
    df_single_batch, model_name="KF single-strike (1s batch)", neg_mass_values=nm_batch
)
rows.append(m_single_batch)
m_single_batch

{'model': 'KF single-strike (1s batch)',
 'ho_nse_mean': 0.5862490365236259,
 'ho_nse_med': 0.30698473869989085,
 'ho_nse_std': 0.8111191249929086,
 'is_nse_mean': 0.25281316720069574,
 'is_nse_med': 0.18498171319574241,
 'is_nse_std': 0.24117442047582988,
 'ho_rmse_mean': 0.00892711870630903,
 'ho_rmse_med': 0.004261351269434499,
 'ho_rmse_std': 0.016671639455144077,
 'is_rmse_mean': 0.004734052494992681,
 'is_rmse_med': 0.0021156218316633613,
 'is_rmse_std': 0.01271690733034955,
 'neg_mass_mean': -0.24232756771820907,
 'neg_mass_med': -0.1948252693898438,
 'arb_frac': 0.99,
 'n_snapshots': 1169482}

## Comparison table

In [22]:
um.comparison_table(rows)

,ho_nse_mean,ho_nse_med,ho_nse_std,is_nse_mean,is_nse_med,is_nse_std,ho_rmse_mean,ho_rmse_med,ho_rmse_std,is_rmse_mean,is_rmse_med,is_rmse_std,neg_mass_mean,neg_mass_med,arb_frac,n_snapshots
model,,,,,,,,,,,,,,,,
SANOS EKF,0.412297,0.369317,0.369471,0.173005,0.165057,0.132913,0.007348,0.005758,0.009006,0.004647,0.003410,0.007868,0.000000,0.000000,0.000000,24066
SANOS 2-stage (DLV fit + KF),1.718257,0.596550,4.562455,0.436169,0.000204,3.065844,0.021370,0.008931,0.043138,0.005097,0.000002,0.026134,0.000000,0.000000,0.000000,24066
KF spline,455.064937,2.100796,1681.596855,460.570114,1.893758,1687.718090,3.741366,0.046643,13.199357,3.754334,0.044584,13.211697,-0.815088,0.000000,0.140727,24066
KF spline,455.064937,2.100796,1681.596855,460.570114,1.893758,1687.718090,3.741366,0.046643,13.199357,3.754334,0.044584,13.211697,-0.029973,0.000000,0.129000,24066
KF single-strike,0.315510,0.199855,0.425689,0.241208,0.183168,0.232976,0.006021,0.002005,0.018334,0.004882,0.001488,0.015072,-0.258785,-0.201179,0.994000,1169635
KF single-strike (1s batch),0.586249,0.306985,0.811119,0.252813,0.184982,0.241174,0.008927,0.004261,0.016672,0.004734,0.002116,0.012717,-0.242328,-0.194825,0.990000,1169482
